In [ ]:
# ETAPA 3 - PASSO 1
# Carregamento do modelo e dos dados de teste

import pandas as pd
import joblib
from pathlib import Path

# Identificar diretório principal do projeto
projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

# Caminhos dos arquivos
arquivo_modelo = projeto / "models" / "random_forest.joblib"
arquivo_limiar = projeto / "models" / "limiar.txt"
arquivo_teste = projeto / "data" / "processed" / "dataset_teste.csv"

# Carregar modelo salvo pelo próprio projeto
modelo = joblib.load(arquivo_modelo)

# Carregar limiar de classificação
limiar = float(
    arquivo_limiar.read_text(encoding="utf-8").strip()
)

# Carregar conjunto de teste
df_teste = pd.read_csv(arquivo_teste)

X_test = df_teste.drop(columns=["Y"])
y_test = df_teste["Y"]

print("=== AVALIAÇÃO FINAL — CARREGAMENTO ===")
print("Modelo:", type(modelo.named_steps["classificador"]).__name__)
print("Limiar:", limiar)
print("Registros de teste:", len(X_test))
print("Variáveis preditoras:", X_test.shape[1])
print("Status: Pronto para avaliação")

### 📝 Observação 01 — Carregamento do Modelo para Avaliação

carregar o modelo Random Forest treinado e o conjunto de teste reservado para a avaliação final.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 2
# Avaliação final do modelo Random Forest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Gerar probabilidades para o conjunto de teste
probabilidades_teste = modelo.predict_proba(X_test)[:, 1]

# Aplicar o limiar escolhido na validação
previsoes_teste = (
    probabilidades_teste >= limiar
).astype(int)

# Calcular métricas finais
accuracy = accuracy_score(y_test, previsoes_teste)
precision = precision_score(y_test, previsoes_teste)
recall = recall_score(y_test, previsoes_teste)
f1 = f1_score(y_test, previsoes_teste)
roc_auc = roc_auc_score(y_test, probabilidades_teste)

print("=== AVALIAÇÃO FINAL — RANDOM FOREST ===")

print(f"Acurácia:  {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-score:  {f1:.2%}")
print(f"ROC-AUC:   {roc_auc:.2%}")

print("\nRegistros avaliados:", len(y_test))
print("Limiar utilizado:", limiar)

### 📝 Observação 02 — Avaliação Final do Random Forest

avaliar a capacidade de generalização do modelo
Random Forest utilizando os 6.000 registros reservados para teste.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 3
# Matriz de confusão final do Random Forest

import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

matriz_final = confusion_matrix(
    y_test,
    previsoes_teste
)

tn, fp, fn, tp = matriz_final.ravel()

print("=== MATRIZ DE CONFUSÃO FINAL ===")

print("Verdadeiros negativos:", tn)
print("Falsos positivos:", fp)
print("Falsos negativos:", fn)
print("Verdadeiros positivos:", tp)

print("\nTotal de registros:", matriz_final.sum())

ConfusionMatrixDisplay(
    confusion_matrix=matriz_final,
    display_labels=["Não inadimplente", "Inadimplente"]
).plot(
    cmap="Blues",
    values_format="d"
)

plt.title("Matriz de Confusão Final — Random Forest")
plt.show()

### 📝 Observação 03 — Matriz de Confusão Final

identificar os acertos e erros do Random Forest
utilizando os 6.000 registros do conjunto de teste final.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 4
# Importância das variáveis do Random Forest

import pandas as pd
import matplotlib.pyplot as plt

# Recuperar componentes do pipeline treinado
preprocessador = modelo.named_steps["preprocessamento"]
classificador = modelo.named_steps["classificador"]

# Obter nomes das variáveis transformadas
nomes_variaveis = preprocessador.get_feature_names_out()

# Importâncias calculadas pelo Random Forest
importancias = pd.DataFrame({
    "Variável": nomes_variaveis,
    "Importância": classificador.feature_importances_
})

importancias = importancias.sort_values(
    by="Importância",
    ascending=False
)

print("=== IMPORTÂNCIA DAS VARIÁVEIS ===")
display(importancias.head(15).round(4))

# Gráfico das 15 variáveis mais importantes
top15 = importancias.head(15).sort_values("Importância")

plt.figure(figsize=(10, 7))

plt.barh(
    top15["Variável"],
    top15["Importância"]
)

plt.title("15 Variáveis Mais Importantes — Random Forest")
plt.xlabel("Importância relativa (MDI)")
plt.ylabel("Variáveis")
plt.tight_layout()
plt.show()

### 📝 Observação 04 — Importância das Variáveis do Random Forest

identificar quais variáveis apresentam maior
importância nas divisões realizadas pelas árvores do modelo.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 5
# Permutation Importance no conjunto de treinamento

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from app.data_split import dividir_dados

# Recuperar o dataset preparado
arquivo_dados = projeto / "data" / "processed" / "dataset_limpo.csv"

df_completo = pd.read_csv(arquivo_dados)

X = df_completo.drop(columns=["Y"])
y = df_completo["Y"]

# Reproduzir a divisão original
X_train, _, y_train, _ = dividir_dados(X, y)

# Calcular importância por permutação
resultado_perm = permutation_importance(
    modelo,
    X_train,
    y_train,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
    n_jobs=1
)

importancias_perm = pd.DataFrame({
    "Variável": X_train.columns,
    "Importância média": resultado_perm.importances_mean,
    "Desvio padrão": resultado_perm.importances_std
}).sort_values(
    by="Importância média",
    ascending=False
)

print("=== PERMUTATION IMPORTANCE ===")
display(importancias_perm.head(15).round(4))

# Gráfico
top10 = importancias_perm.head(10).sort_values(
    "Importância média"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top10["Variável"],
    top10["Importância média"]
)

plt.title("Permutation Importance — Random Forest")
plt.xlabel("Redução média na ROC-AUC")
plt.tight_layout()
plt.show()

### 📝 Observação 05 — Análise de Permutation Importance

investigar a importância das variáveis
financeiras por meio da alteração do desempenho
preditivo do Random Forest.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 6
# Curvas ROC e Precision-Recall

import matplotlib.pyplot as plt

from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Curva ROC
RocCurveDisplay.from_predictions(
    y_test,
    probabilidades_teste
)

plt.title("Curva ROC — Random Forest")
plt.grid(alpha=0.3)
plt.show()

# Curva Precision-Recall
PrecisionRecallDisplay.from_predictions(
    y_test,
    probabilidades_teste
)

plt.title("Curva Precision-Recall — Random Forest")
plt.grid(alpha=0.3)
plt.show()

### 📝 Observação 06 — Curvas ROC e Precision-Recall

avaliar a capacidade de discriminação do
Random Forest e o equilíbrio entre Precision e Recall
utilizando diferentes limiares de classificação.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 7
# Análise de calibração das probabilidades

import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# Calcular curva de calibração
fracao_positivos, probabilidade_media = calibration_curve(
    y_test,
    probabilidades_teste,
    n_bins=10,
    strategy="quantile"
)

# Calcular Brier Score
brier = brier_score_loss(
    y_test,
    probabilidades_teste
)

print("=== CALIBRAÇÃO DAS PROBABILIDADES ===")
print(f"Brier Score: {brier:.4f}")

plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1],
    [0, 1],
    "--",
    label="Calibração perfeita"
)

plt.plot(
    probabilidade_media,
    fracao_positivos,
    marker="o",
    label="Random Forest"
)

plt.xlabel("Probabilidade média prevista")
plt.ylabel("Fração real de inadimplentes")
plt.title("Curva de Calibração — Random Forest")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 📝 Observação 07 — Calibração das Probabilidades

verificar se as probabilidades estimadas
pelo Random Forest correspondem às frequências reais
de inadimplência.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 8
# Comparação da calibração com baseline probabilístico

import numpy as np
from sklearn.metrics import brier_score_loss

# Prevalência histórica conhecida no conjunto de treinamento
arquivo_dados = projeto / "data" / "processed" / "dataset_limpo.csv"
df_completo = pd.read_csv(arquivo_dados)

from app.data_split import dividir_dados

_, _, y_train, _ = dividir_dados(df_completo.drop(columns=["Y"]), df_completo["Y"])

prevalencia_treino = y_train.mean()

# Baseline: mesma probabilidade para todos os registros
probabilidades_baseline = np.full(
    len(y_test),
    prevalencia_treino
)

brier_baseline = brier_score_loss(
    y_test,
    probabilidades_baseline
)

brier_modelo = brier_score_loss(
    y_test,
    probabilidades_teste
)

print("=== COMPARAÇÃO DO BRIER SCORE ===")
print(f"Prevalência no treinamento: {prevalencia_treino:.2%}")
print(f"Brier Score - Baseline: {brier_baseline:.4f}")
print(f"Brier Score - Random Forest: {brier_modelo:.4f}")
print(f"Redução relativa do erro: {(1 - brier_modelo / brier_baseline):.2%}")

### 📝 Observação 08 — Comparação do Brier Score com o Baseline

comparar a qualidade das probabilidades
estimadas pelo Random Forest com um modelo de referência
baseado na prevalência histórica de inadimplência.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 9
# Exportação dos resultados de previsão

import pandas as pd
from pathlib import Path

# Criar tabela de resultados
resultados_finais = pd.DataFrame({
    "ID_Registro": X_test.index,
    "Inadimplencia_Real": y_test.to_numpy(),
    "Probabilidade_Inadimplencia": probabilidades_teste,
    "Inadimplencia_Prevista": previsoes_teste
})

# Classificação experimental usando o limiar validado
resultados_finais["Classificacao"] = (
    resultados_finais["Inadimplencia_Prevista"]
    .map({0: "Não sinalizado", 1: "Sinalizado"})
)

# Organizar diretório de saída
pasta_saida = projeto / "data" / "processed"
pasta_saida.mkdir(parents=True, exist_ok=True)

arquivo_resultados = pasta_saida / "previsoes_inadimplencia.csv"

resultados_finais.to_csv(
    arquivo_resultados,
    index=False
)

print("=== EXPORTAÇÃO DOS RESULTADOS ===")
print("Arquivo:", arquivo_resultados)
print("Registros exportados:", len(resultados_finais))
print("Colunas:", resultados_finais.shape[1])

display(resultados_finais.head())

### 📝 Observação 09 — Exportação dos Resultados de Machine Learning

exportar as previsões do Random Forest para
um arquivo CSV que poderá ser utilizado pelo Excel e n8n.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# ETAPA 3 - PASSO 10
# Resumo dos resultados para Excel

import pandas as pd

total = len(resultados_finais)

sinalizados = (
    resultados_finais["Inadimplencia_Prevista"] == 1
).sum()

nao_sinalizados = total - sinalizados

resumo = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Clientes sinalizados",
        "Clientes não sinalizados",
        "Percentual sinalizado",
        "Probabilidade média",
        "Probabilidade mínima",
        "Probabilidade máxima"
    ],
    "Valor": [
        total,
        sinalizados,
        nao_sinalizados,
        f"{sinalizados / total:.2%}",
        f"{resultados_finais['Probabilidade_Inadimplencia'].mean():.2%}",
        f"{resultados_finais['Probabilidade_Inadimplencia'].min():.2%}",
        f"{resultados_finais['Probabilidade_Inadimplencia'].max():.2%}"
    ]
})

print("=== RESUMO DOS RESULTADOS ===")
display(resumo)

### 📝 Observação 10 — Resumo dos Resultados para Excel

consolidar os principais indicadores das
previsões produzidas pelo modelo Random Forest.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# Exportação dos resultados para Excel

%pip install openpyxl

In [ ]:
# ETAPA 3 - PASSO 12
# Exportação das previsões e indicadores para Excel

import pandas as pd
from pathlib import Path

# Pasta principal do projeto
projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

# Diretório de saída
pasta_saida = projeto / "data" / "processed"
pasta_saida.mkdir(parents=True, exist_ok=True)

# Arquivo Excel
arquivo_excel = pasta_saida / "Resultado_Inadimplencia.xlsx"

# Criar planilha com duas abas
with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

    resultados_finais.to_excel(
        writer,
        sheet_name="Previsões",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="Indicadores",
        index=False
    )

print("=== EXPORTAÇÃO PARA EXCEL ===")
print("Arquivo:", arquivo_excel)
print("Abas: Previsões e Indicadores")
print("Registros exportados:", len(resultados_finais))
print("Status: Exportação concluída")

### 📝 Observação 12 — Exportação dos Resultados para Excel

disponibilizar os resultados do modelo
Random Forest em uma planilha Excel.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


### Observação 5 — Validação do Random Forest via FastAPI

verificar se o modelo disponibilizado pela
FastAPI produz resultados equivalentes aos obtidos
diretamente no notebook Jupyter.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.
